In [ ]:
from google.colab import drive
drive.mount('/content/drive')


# Geometry-V4 G0/G1 fixed SD3.5 run
This notebook accepts only a controller-supplied immutable source bundle; it does not clone a Git branch. G0: 5101--5104 identity. G1 holdout: 6101--6104 x identity/rotation_5/scale_0.9/translation_0.08_0/crop_0.9.


In [ ]:
from pathlib import Path
import hashlib, json, tarfile
DRIVE_ROOT = Path('/content/drive/MyDrive/CEG-WM/Geometry-V4-G0-G1')
manifest = json.loads((DRIVE_ROOT/'source'/'controller_manifest.json').read_text(encoding='ascii'))
SOURCE_BUNDLE = DRIVE_ROOT / 'source' / manifest['bundle_filename']
SOURCE_BUNDLE_SHA256 = manifest['bundle_sha256']
SOURCE_EXACT = manifest['source_exact']
RUN_ROOT = DRIVE_ROOT / 'runs' / SOURCE_EXACT
assert isinstance(SOURCE_BUNDLE_SHA256, str) and len(SOURCE_BUNDLE_SHA256) == 64
assert isinstance(SOURCE_EXACT, str) and len(SOURCE_EXACT) == 40
got = hashlib.sha256(SOURCE_BUNDLE.read_bytes()).hexdigest()
assert got == SOURCE_BUNDLE_SHA256, (got, SOURCE_BUNDLE_SHA256)
RUN_ROOT.mkdir(parents=True, exist_ok=False)
with tarfile.open(SOURCE_BUNDLE) as archive: archive.extractall('/content/geometry-v4-g0-g1')
REPO = Path('/content/geometry-v4-g0-g1')
CONFIG_SHA256 = hashlib.sha256((REPO/'configs/geometry_v4/geometry_v4_g0_g1_v1.json').read_bytes()).hexdigest()
print({'source_exact': SOURCE_EXACT, 'bundle_sha256': got, 'config_sha256': CONFIG_SHA256, 'model': 'stabilityai/stable-diffusion-3.5-medium', 'scheduler': 'pipeline default, recorded at runtime', 'steps': 20, 'dtype': 'float16', 'placement': 'callback step 19 final latent before VAE decode', 'artifact_root': str(RUN_ROOT)})


In [ ]:
%cd /content/geometry-v4-g0-g1
!pip install -q -e .
from google.colab import userdata
from experiments.geometry_v4_generative_engine import run
HF_TOKEN = userdata.get('HF_TOKEN')
DETECTION_KEY = userdata.get('CEGWM_DETECTION_KEY')
WRONG_KEY = userdata.get('CEGWM_WRONG_KEY')
assert all(isinstance(v, str) and v for v in (HF_TOKEN, DETECTION_KEY, WRONG_KEY))
# Controller must bind the existing unchanged content detector: (current_rgb, normalized_key_bytes) -> scalar.
# It must not close over original RGB, latent, writer residual, truth, or attack metadata.
unchanged_content_detector = None
assert callable(unchanged_content_detector), 'bind the existing RGB/key-only content detector before G0'
# G0 only. A re-run requires a new writer commit and replacement verified bundle.
g0_records = run('G0', DETECTION_KEY, WRONG_KEY, repo_root=REPO, hf_token=HF_TOKEN, content_detector=unchanged_content_detector, artifact_root=RUN_ROOT)
print({'g0_units': len(g0_records), 'g0_passed': sum(r['final_rgb'] is not None and r['final_rgb']['passed'] for r in g0_records)})
# Do not run G1 here. After a separately reviewed 4/4 G0 freeze, execute the identical call with stage='G1' exactly once in a new create-only RUN_ROOT.
